# PhoW2V and PhoBERT + PhoW2V - 5-fold cross-validation (Kaggle)

**Before running**

1. Accelerator **GPU**, Internet **On**. Code and processed data are cloned from GitHub (push your latest changes first).
2. **Attach a dataset containing the PhoW2V word-level vectors**, e.g. `word2vec_vi_words_300dims.txt` (Add Input -> search "phow2v"). The official download host is offline; the authors' Google Drive mirror is linked from https://github.com/datquocnguyen/PhoW2V. Use the **word-level** file, not the syllable one, because the text is word-segmented.

Three models are trained on the same 5 folds as the earlier PhoBERT runs, so every out-of-fold score is comparable:

- `phobert`: PhoBERT alone (the regularised settings), as the reference
- `phow2v`: PhoW2V embeddings + BiLSTM
- `phobert_phow2v`: one model with both encoders, features concatenated

A fourth row averages the probabilities of the separate `phobert` and `phow2v` models (late fusion).

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_phow2v_cv.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_phow2v_cv.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/phow2v_cv")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "phow2v_cv"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Locate the PhoW2V file

In [ ]:
from architecture.phow2v import find_phow2v

W2V_PATH = find_phow2v("")          # or pass the file path explicitly
print(W2V_PATH, f"{W2V_PATH.stat().st_size / 1e9:.2f} GB")

## Train

Roughly 30 minutes on a T4: about 12 for `phobert`, 3 for `phow2v`, 14 for `phobert_phow2v`. Remove an entry from `RUNS` to skip it (late fusion needs both `phobert` and `phow2v`).

In [ ]:
from trainer import train_phobert_cv, train_phow2v_cv
from trainer.train_phobert_cv import CVConfig
from trainer.train_phow2v_cv import W2VConfig

BASE = dict(
    data_dir=str(DATA_DIR),
    n_folds=5,
    max_length=128,
    use_lead=True,
    batch_size=32,
    weighted_loss=True,
    patience=3,
    monitor="loss",
    seed=42,
    keep_checkpoints=False,
    limit=0,              # set to e.g. 64 for a quick smoke test
)
W2V = dict(w2v_path=str(W2V_PATH), w2v_hidden=128, w2v_dropout=0.3, w2v_freeze=True)
RUNS = {
    "phobert": CVConfig(**BASE, lr=2e-5, epochs=5, rdrop_alpha=1.0, layer_decay=0.9),
    "phow2v": W2VConfig(**BASE, **W2V, mode="phow2v", lr=1e-3, epochs=15),
    "phobert_phow2v": W2VConfig(**BASE, **W2V, mode="phobert_phow2v", lr=2e-5, w2v_lr=1e-3,
                                epochs=5, rdrop_alpha=1.0, layer_decay=0.9),
}

results = {}
for name, cfg in RUNS.items():
    print(f"################ {name} ################")
    cfg.out_dir = str(OUT_DIR / name)
    runner = train_phow2v_cv if isinstance(cfg, W2VConfig) else train_phobert_cv
    results[name] = runner.run(cfg)

## Late fusion: average the separate PhoBERT and PhoW2V models

The mixing weight is chosen on the out-of-fold predictions, then applied to the test ensemble.

In [ ]:
import numpy as np
import pandas as pd
from trainer.trainer import classification_metrics

def f1_at(table, probs, threshold=0.5):
    return classification_metrics((table["label"] == "clickbait").to_numpy(), (probs >= threshold).to_numpy())

late = None
if {"phobert", "phow2v"} <= set(results):
    def paired(key):
        a, b = results["phobert"][key], results["phow2v"][key]
        return a.merge(b[["id", "prob_clickbait"]], on="id", suffixes=("_phobert", "_phow2v"), validate="one_to_one")

    oof_pair, test_pair = paired("oof_predictions"), paired("test_predictions")
    weights = pd.DataFrame(
        {"oof_f1": [f1_at(oof_pair, w * oof_pair["prob_clickbait_phobert"] + (1 - w) * oof_pair["prob_clickbait_phow2v"])["f1"]
                    for w in np.arange(0, 1.01, 0.1)]},
        index=pd.Index(np.arange(0, 1.01, 0.1).round(1), name="phobert_weight"),
    )
    W = weights["oof_f1"].idxmax()
    mix = lambda t: W * t["prob_clickbait_phobert"] + (1 - W) * t["prob_clickbait_phow2v"]
    late = {"weight": W, "oof": f1_at(oof_pair, mix(oof_pair)), "test": f1_at(test_pair, mix(test_pair))}
    print("best PhoBERT weight on out-of-fold:", W)
    display(weights.round(4).T)
    print("agreement between the two models (out-of-fold):",
          round(((oof_pair["prob_clickbait_phobert"] >= 0.5) == (oof_pair["prob_clickbait_phow2v"] >= 0.5)).mean(), 3))

## Compare

Judge by `oof_f1` (3,067 articles). Earlier PhoBERT runs scored 0.766-0.774 out-of-fold; differences under about one point are noise.

In [ ]:
rows = {}
for name, r in results.items():
    s = r["summary"]
    rows[name] = {"oof_f1": s["oof"]["f1"], "oof_precision": s["oof"]["precision"], "oof_recall": s["oof"]["recall"],
                  "single_test_f1": s["single_model_test_mean"]["f1"], "ensemble_test_f1": s["ensemble_test"]["f1"]}
if late:
    rows[f"late fusion (w={late['weight']})"] = {"oof_f1": late["oof"]["f1"], "oof_precision": late["oof"]["precision"],
                                               "oof_recall": late["oof"]["recall"], "ensemble_test_f1": late["test"]["f1"]}
comparison = pd.DataFrame(rows).T
display(comparison.round(4))

SHOW = "phobert_phow2v" if "phobert_phow2v" in results else list(results)[-1]   # run detailed below
result = results[SHOW]
print("showing details for:", SHOW)

## Per-fold results

`val_*` is each model on its own held-out fold; `test_*` is the same model alone on the test split.

In [ ]:
folds = result["fold_metrics"]
display(folds.round(4))
folds.agg(["mean", "std"]).round(4)

## Cross-validated score vs single model vs ensemble

In [ ]:
summary = pd.DataFrame(result["summary"]).T.loc[["oof", "single_model_test_mean", "single_model_test_std", "ensemble_test"]]
summary.index = ["out-of-fold (pooled train+val)", "single model on test (mean)", "single model on test (std)", "ensemble on test"]
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([f"fold {i}" for i in folds.index], folds["test_f1"], color="#9ecae1", label="single model")
ax.axhline(result["summary"]["ensemble_test"]["f1"], color="#E45756", linestyle="--", label="ensemble")
ax.set_ylim(max(0, folds["test_f1"].min() - 0.1), 1)
ax.set_ylabel("test F1 (clickbait)")
ax.legend(frameon=False)
fig.tight_layout()

## Ensemble on test

In [ ]:
test = result["test_predictions"]
pd.crosstab(test["label"], test["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

### Threshold

Chosen on the out-of-fold predictions (never on test), then applied to the ensemble.

In [ ]:
oof = result["oof_predictions"]
oof_y, test_y = (oof["label"] == "clickbait").to_numpy(), (test["label"] == "clickbait").to_numpy()

rows = []
for t in np.arange(0.2, 0.81, 0.05):
    o = classification_metrics(oof_y, (oof["prob_clickbait"] >= t).to_numpy())
    e = classification_metrics(test_y, (test["prob_clickbait"] >= t).to_numpy())
    rows.append({"threshold": round(t, 2), "oof_precision": o["precision"], "oof_recall": o["recall"], "oof_f1": o["f1"],
                 "test_precision": e["precision"], "test_recall": e["recall"], "test_f1": e["f1"]})
thresholds = pd.DataFrame(rows).set_index("threshold")
print("best threshold on out-of-fold F1:", thresholds["oof_f1"].idxmax())
thresholds.round(3)

### Out-of-fold results per source and category

Computed on all pooled articles, so the groups are large enough to compare.

In [ ]:
def by_group(table, col):
    rows = {}
    for name, g in table.groupby(col):
        m = classification_metrics((g["label"] == "clickbait").to_numpy(), (g["pred"] == "clickbait").to_numpy())
        rows[name] = {"n": len(g), "clickbait_rate": (g["label"] == "clickbait").mean(), **m}
    return pd.DataFrame(rows).T.round(3).sort_values("n", ascending=False)

by_group(oof, "source")

In [ ]:
by_group(oof, "category")

### Most confident out-of-fold mistakes

Each prediction comes from a model that never saw the article, so confident disagreements are good candidates for a label review.

In [ ]:
pd.set_option("display.max_colwidth", 140)
wrong = oof[oof["label"] != oof["pred"]].copy()
wrong["confidence"] = (wrong["prob_clickbait"] - 0.5).abs() + 0.5
print(len(wrong), "out-of-fold mistakes;", (wrong["confidence"] >= 0.9).sum(), "with confidence >= 0.9")
wrong.sort_values("confidence", ascending=False).head(30)[["id", "source", "title", "label", "pred", "prob_clickbait"]]

## Saved files

`summary.json`, `fold_metrics.csv`, `oof_predictions.csv`, `test_predictions.csv`, `tokenizer/`, per-fold `history.json`. On Kaggle they are under `/kaggle/working/phow2v_cv/<run name>` and appear in the Output tab after **Save Version**.

In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")